# SBP-DG Table 2 Corrected Ablation Study

## Alignment with the current Simplex projected-SBP formulation

This notebook is a corrected version of the previous `test3.ipynb`. The production
RHS and all diagnostics use the same operator construction.

The corrected **Group 0 (Full Alignment)** uses:

1. **H1-A — projected metric trace**
   \[
   c_r^b = E c_r,\qquad c_s^b = E c_s.
   \]

2. **H2-A — volume-compatible interior boundary flux**
   \[
   F_{\mathrm{cons}}^- = n_r E(c_rq)+n_s E(c_sq),
   \]
   and, for the three-term split form,
   \[
   F_{\mathrm{split}}^-=
   \frac12F_{\mathrm{cons}}^-+\frac12a_Mq_M.
   \]

3. **H3-A — interface-single-valued line velocity**
   \[
   a_{\mathrm{common}}=\frac12(a_M-a_P),
   \]
   where \(a_M\) and \(a_P\) are formed in their own local reference
   coordinates and then paired using the face orientation map. The local speed
   \(a_M\) is retained in the split interior flux; the common speed is used only
   by the numerical flux.

4. **H4-A — correctly scaled projected lift**
   \[
   H=|T|W,\qquad M=V^THV,\qquad
   L_f=VM^{-1}V_f^TW_f,\qquad |T|=2.
   \]

The strong-form surface correction uses
\[
q_t=-\frac{1}{J}D F+\frac{1}{J}L_f(F^- - F^*).
\]

The notebook also reports the operator-level mass residual
\[
\|m^TA\|_\infty,
\]
which is stronger than checking \(m^TAq_0\) for one initial state.

> This remains a research diagnostic notebook. It clones the external
> `wcw100168/Simplex-DG-solver` package for basis/quadrature helpers and does
> not replace the tested implementation in `NKRlyq1213/Simplex-DG-solver`.


In [1]:
# STREAMING_CHUNK: Configuring environment and dependencies...
import sys
import os
import gc
import time
import subprocess
from pathlib import Path

import numpy as np
import scipy.linalg as la
import pandas as pd

# Avoid repeated os.chdir("Simplex-DG-solver"), which previously created
# nested paths when this cell was executed more than once.
START_DIR = Path.cwd().resolve()
REPO_NAME = "Simplex-DG-solver"
REPO_URL = "https://github.com/wcw100168/Simplex-DG-solver.git"

if START_DIR.name == REPO_NAME:
    REPO_DIR = START_DIR
else:
    REPO_DIR = START_DIR / REPO_NAME

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)],
    check=True,
)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    GIT_COMMIT = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    GIT_COMMIT = "unknown"

os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import jit
from functools import partial

print(">>> Repository:", REPO_DIR)
print(">>> Git commit:", GIT_COMMIT)
print(">>> Active JAX Devices:", jax.devices())

from src.core.generators import get_reference_data
from src.core.connectivity import build_connectivity
from src.bases.vandermonde import (
    vandermonde_2d_dubiner,
    grad_vandermonde_2d_dubiner,
)


Obtaining file:///Users/user/Downloads/%E5%B0%88%E9%A1%8C%E7%9F%AD%E8%AC%9B/0727/Simplex-DG-solver
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for simplex-dg-solver (pyproject.toml): started
  Building editable for simplex-dg-solver (pyproject.toml): finished with status 'done'
  Created wheel for simplex-dg-solver: filename=simplex_dg_solver-0.2.0-0.editable-py3-none-any.whl size=2925 sha256=fa9e783be2f1b5ca6d03cd17a697a2359d364283faf10757afe8634c78dd238c
  Stored in directory: /private/var/folders/xw/460kbmwd03b4w67v8f_q

In [2]:
# STREAMING_CHUNK:Defining geometry and operator builders...
def generate_spherical_octahedron_mesh(n_div: int, R_sphere: float = 1.0):
    v = np.array([
        [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]
    ], dtype=float)
    base_faces = [[0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4], [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5]]
    nodes, EToV, node_map = [], [], {}

    def get_node_id(pt):
        pt = np.array(pt)
        pt = R_sphere * pt / np.linalg.norm(pt)
        key = (round(pt[0], 8), round(pt[1], 8), round(pt[2], 8))
        if key not in node_map:
            node_map[key] = len(nodes)
            nodes.append(pt.tolist())
        return node_map[key]

    for face in base_faces:
        v0, v1, v2 = v[face[0]], v[face[1]], v[face[2]]
        for i in range(n_div):
            for j in range(n_div - i):
                p1 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * (j / n_div)
                p2 = v0 + (v1 - v0) * ((i + 1) / n_div) + (v2 - v0) * (j / n_div)
                p3 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * ((j + 1) / n_div)
                EToV.append([get_node_id(p1), get_node_id(p2), get_node_id(p3)])
                if i > 0:
                    p4 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * (j / n_div)
                    p5 = v0 + (v1 - v0) * ((i - 1) / n_div) + (v2 - v0) * ((j + 1) / n_div)
                    p6 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * ((j + 1) / n_div)
                    EToV.append([get_node_id(p4), get_node_id(p6), get_node_id(p5)])
    return np.array(nodes), np.array(EToV)

def get_face_quadrature_1d(k_degree):
    n_points = k_degree + 1
    points_1d, weights_1d = np.polynomial.legendre.leggauss(n_points)
    r1, s1 = points_1d, -np.ones_like(points_1d)
    r2, s2 = -points_1d, points_1d
    r3, s3 = -np.ones_like(points_1d), -points_1d
    xi_face = np.concatenate([r1, r2, r3])
    eta_face = np.concatenate([s1, s2, s3])
    return points_1d, weights_1d, xi_face, eta_face

_LOCAL_FACE_VERTICES = np.array(
    [
        [0, 1],  # bottom:   v1 -> v2
        [1, 2],  # diagonal: v2 -> v3
        [2, 0],  # left:     v3 -> v1
    ],
    dtype=int,
)


def build_face_global_index_maps(EToV, EToE, EToF, K, nfp):
    """Build oriented face maps from the actual shared-vertex ordering."""
    EToV = np.asarray(EToV, dtype=int)
    vmapM = np.zeros((3 * nfp, K), dtype=int)
    vmapP = np.zeros((3 * nfp, K), dtype=int)
    is_boundary = np.zeros((3 * nfp, K), dtype=bool)

    for k_elem in range(K):
        for face in range(3):
            face_idx_M = np.arange(face * nfp, (face + 1) * nfp)
            vmapM[face_idx_M, k_elem] = (
                k_elem * (3 * nfp) + face_idx_M
            )

            k_neighbor = int(EToE[k_elem, face])
            f_neighbor = int(EToF[k_elem, face])

            if k_neighbor == k_elem:
                vmapP[face_idx_M, k_elem] = vmapM[face_idx_M, k_elem]
                is_boundary[face_idx_M, k_elem] = True
                continue

            face_idx_P = np.arange(
                f_neighbor * nfp,
                (f_neighbor + 1) * nfp,
            )

            va, vb = EToV[k_elem, _LOCAL_FACE_VERTICES[face]]
            vc, vd = EToV[k_neighbor, _LOCAL_FACE_VERTICES[f_neighbor]]

            if va == vc and vb == vd:
                ordered_face_idx_P = face_idx_P
            elif va == vd and vb == vc:
                ordered_face_idx_P = face_idx_P[::-1]
            else:
                raise ValueError(
                    "Paired faces do not have consistent oriented vertex IDs: "
                    f"(k,f)=({k_elem},{face}), "
                    f"neighbor=({k_neighbor},{f_neighbor}), "
                    f"local edge=({va},{vb}), "
                    f"neighbor edge=({vc},{vd})."
                )

            vmapP[face_idx_M, k_elem] = (
                k_neighbor * (3 * nfp) + ordered_face_idx_P
            )

    return vmapM, vmapP, is_boundary

def compute_global_coordinates(nodes, EToV, xi_ref, eta_ref, R_sphere):
    K, Np = EToV.shape[0], len(xi_ref)
    X, Y, Z = np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K))
    L1, L2, L3 = -(xi_ref + eta_ref) / 2.0, (xi_ref + 1.0) / 2.0, (eta_ref + 1.0) / 2.0
    for k in range(K):
        v1, v2, v3 = nodes[EToV[k]]
        x_k = L1 * v1[0] + L2 * v2[0] + L3 * v3[0]
        y_k = L1 * v1[1] + L2 * v2[1] + L3 * v3[1]
        z_k = L1 * v1[2] + L2 * v2[2] + L3 * v3[2]
        r = np.sqrt(x_k**2 + y_k**2 + z_k**2)
        X[:, k], Y[:, k], Z[:, k] = R_sphere * (x_k / r), R_sphere * (y_k / r), R_sphere * (z_k / r)
    return X, Y, Z

def compute_exact_metrics_3d(nodes, EToV, xi_ref, eta_ref, xi_face, eta_face):
    K, Np = EToV.shape[0], len(xi_ref)
    n_face_pts = len(xi_face)
    J_exact = np.zeros((Np, K))
    a1x, a1y, a1z = np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K))
    a2x, a2y, a2z = np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K))
    exact_c_r_b, exact_c_s_b = np.zeros((n_face_pts, K)), np.zeros((n_face_pts, K))
    
    L1, L2, L3 = -(xi_ref + eta_ref) / 2.0, (xi_ref + 1.0) / 2.0, (eta_ref + 1.0) / 2.0
    L1_f, L2_f, L3_f = -(xi_face + eta_face) / 2.0, (xi_face + 1.0) / 2.0, (eta_face + 1.0) / 2.0
    u0, alpha0 = (2 * np.pi * 1.0) / (12 * 86400), np.pi / 4.0
    Omega_x, Omega_y, Omega_z = -np.sin(alpha0) * u0, 0.0, np.cos(alpha0) * u0
    
    for k in range(K):
        v1, v2, v3 = nodes[EToV[k]]
        x_flat = L1 * v1[0] + L2 * v2[0] + L3 * v3[0]
        y_flat = L1 * v1[1] + L2 * v2[1] + L3 * v3[1]
        z_flat = L1 * v1[2] + L2 * v2[2] + L3 * v3[2]
        norm_x = np.sqrt(x_flat**2 + y_flat**2 + z_flat**2)
        dr_x, dr_y, dr_z = -0.5 * v1[0] + 0.5 * v2[0], -0.5 * v1[1] + 0.5 * v2[1], -0.5 * v1[2] + 0.5 * v2[2]
        ds_x, ds_y, ds_z = -0.5 * v1[0] + 0.5 * v3[0], -0.5 * v1[1] + 0.5 * v3[1], -0.5 * v1[2] + 0.5 * v3[2]
        cross_x, cross_y, cross_z = dr_y * ds_z - dr_z * ds_y, dr_z * ds_x - dr_x * ds_z, dr_x * ds_y - dr_y * ds_x
        h = v1[0] * cross_x + v1[1] * cross_y + v1[2] * cross_z
        J_exact[:, k] = h / (norm_x**3)
        factor = norm_x / h
        a1x[:, k], a1y[:, k], a1z[:, k] = factor * (ds_y * z_flat - ds_z * y_flat), factor * (ds_z * x_flat - ds_x * z_flat), factor * (ds_x * y_flat - ds_y * x_flat)
        a2x[:, k], a2y[:, k], a2z[:, k] = factor * (y_flat * dr_z - z_flat * dr_y), factor * (z_flat * dr_x - x_flat * dr_z), factor * (x_flat * dr_y - y_flat * dr_x)
        
        # Face exact calculation
        xf, yf, zf = L1_f * v1[0] + L2_f * v2[0] + L3_f * v3[0], L1_f * v1[1] + L2_f * v2[1] + L3_f * v3[1], L1_f * v1[2] + L2_f * v2[2] + L3_f * v3[2]
        norm_xf = np.sqrt(xf**2 + yf**2 + zf**2)
        J_bf = h / (norm_xf**3)
        factor_f = norm_xf / h
        a1x_f, a1y_f, a1z_f = factor_f * (ds_y * zf - ds_z * yf), factor_f * (ds_z * xf - ds_x * zf), factor_f * (ds_x * yf - ds_y * xf)
        a2x_f, a2y_f, a2z_f = factor_f * (yf * dr_z - zf * dr_y), factor_f * (zf * dr_x - xf * dr_z), factor_f * (xf * dr_y - yf * dr_x)
        U_f, V_f, W_f = Omega_y * zf - Omega_z * yf, Omega_z * xf - Omega_x * zf, Omega_x * yf - Omega_y * xf
        exact_c_r_b[:, k] = J_bf * (a1x_f * U_f + a1y_f * V_f + a1z_f * W_f)
        exact_c_s_b[:, k] = J_bf * (a2x_f * U_f + a2y_f * V_f + a2z_f * W_f)
        
    return J_exact, a1x, a1y, a1z, a2x, a2y, a2z, exact_c_r_b, exact_c_s_b

def compute_h_min(nodes, EToV):
    vertices = nodes[EToV]
    e01 = np.linalg.norm(vertices[:, 1, :] - vertices[:, 0, :], axis=1)
    e12 = np.linalg.norm(vertices[:, 2, :] - vertices[:, 1, :], axis=1)
    e20 = np.linalg.norm(vertices[:, 0, :] - vertices[:, 2, :], axis=1)
    return np.minimum(np.minimum(e01, e12), e20)

In [3]:
# STREAMING_CHUNK:Defining velocity and initial conditions...
def velocity_solid_body_3d(X, Y, Z, u0, alpha0):
    Omega_x, Omega_y, Omega_z = -np.sin(alpha0) * u0, 0.0, np.cos(alpha0) * u0
    return Omega_y * Z - Omega_z * Y, Omega_z * X - Omega_x * Z, Omega_x * Y - Omega_y * X

@jit
def exact_gaussian_bell_3d_jax(X, Y, Z, t, u0, R, alpha0=0.0):
    Omega = u0 / R
    theta_rot = -Omega * t
    Kx, Ky, Kz = -jnp.sin(alpha0), 0.0, jnp.cos(alpha0)
    dot_KV = Kx * X + Ky * Y + Kz * Z
    KxV_x, KxV_y, KxV_z = Ky * Z - Kz * Y, Kz * X - Kx * Z, Kx * Y - Ky * X
    X_rot = X * jnp.cos(theta_rot) + KxV_x * jnp.sin(theta_rot) + Kx * dot_KV * (1 - jnp.cos(theta_rot))
    Y_rot = Y * jnp.cos(theta_rot) + KxV_y * jnp.sin(theta_rot) + Ky * dot_KV * (1 - jnp.cos(theta_rot))
    Z_rot = Z * jnp.cos(theta_rot) + KxV_z * jnp.sin(theta_rot) + Kz * dot_KV * (1 - jnp.cos(theta_rot))
    dot_product = ((X_rot/R) * 0.0 + (Y_rot/R) * 1.0 + (Z_rot/R) * 0.0)
    dist = R * jnp.arccos(jnp.clip(dot_product, -1.0, 1.0))
    return jnp.exp(-10.0 * (dist / R)**2)

In [4]:
# STREAMING_CHUNK: Configuring corrected JAX RHS evaluator...
@partial(
    jit,
    static_argnames=[
        "flux_type",
        "formulation",
        "h1_mode",
        "h2_mode",
        "h3_mode",
        "h4_mode",
    ],
)
def compute_rhs_ablation(
    q,
    D_r_ref,
    D_s_ref,
    E,
    L_f,
    J,
    vmapM,
    vmapP,
    face_is_boundary,
    face_w,
    projected_lift_core,
    c_r,
    c_s,
    exact_c_r_b,
    exact_c_s_b,
    flux_type="central",
    formulation="split3",
    h1_mode="A",
    h2_mode="A",
    h3_mode="A",
    h4_mode="A",
):
    """Evaluate the corrected semi-discrete RHS.

    Conventions
    -----------
    penalty_b = F_minus - F_star
    RHS        = -volume_divergence + J^{-1} L penalty_b

    H3-A constructs a common interface line velocity from the two oriented
    scalar line velocities, not by averaging local contravariant components.
    """
    Np, K = q.shape
    nfp = len(face_w) // 3

    # Reference line-flux coefficients for the three [-1,1]-parameterized
    # faces: bottom, diagonal, left.
    nr_f = jnp.concatenate(
        [jnp.zeros(nfp), jnp.ones(nfp), -jnp.ones(nfp)]
    )[:, jnp.newaxis]
    ns_f = jnp.concatenate(
        [-jnp.ones(nfp), jnp.ones(nfp), jnp.zeros(nfp)]
    )[:, jnp.newaxis]

    # 1. Volume term
    div_r = D_r_ref @ (c_r * q)
    div_s = D_s_ref @ (c_s * q)

    if formulation == "divergence":
        volume_term = -(div_r + div_s) / J
    elif formulation == "split3":
        adv_r = c_r * (D_r_ref @ q)
        adv_s = c_s * (D_s_ref @ q)
        cor_r = q * (D_r_ref @ c_r)
        cor_s = q * (D_s_ref @ c_s)
        volume_term = -0.5 * (
            div_r + div_s + adv_r + adv_s + cor_r + cor_s
        ) / J
    elif formulation == "split2":
        adv_r = c_r * (D_r_ref @ q)
        adv_s = c_s * (D_s_ref @ q)
        volume_term = -0.5 * (
            div_r + div_s + adv_r + adv_s
        ) / J
    else:
        raise ValueError(
            "formulation must be 'divergence', 'split2', or 'split3'."
        )

    # 2. H1: metric trace
    if h1_mode == "A":
        c_r_b = E @ c_r
        c_s_b = E @ c_s
    else:
        c_r_b = exact_c_r_b
        c_s_b = exact_c_s_b

    # Local outward line velocity on every local face.
    a_local_b = nr_f * c_r_b + ns_f * c_s_b
    a_local_flat = a_local_b.T.flatten()
    a_M = a_local_flat[vmapM]
    a_P = a_local_flat[vmapP]

    # 3. H3: common numerical-flux speed
    if h3_mode == "A":
        # a_P uses the neighbor's own outward orientation.
        # Re-express it in the current minus-side orientation by subtraction.
        a_common = 0.5 * (a_M - a_P)
        a_star = jnp.where(face_is_boundary, a_M, a_common)
    else:
        # Deliberately one-sided ablation mode.
        a_star = a_M

    # State traces.
    q_b = E @ q
    q_b_flat = q_b.T.flatten()
    q_M = q_b_flat[vmapM]
    q_P = q_b_flat[vmapP]

    flux_key = flux_type.lower()
    if flux_key == "central":
        F_star = 0.5 * a_star * (q_M + q_P)
    elif flux_key == "upwind":
        F_star = (
            0.5 * a_star * (q_M + q_P)
            - 0.5 * jnp.abs(a_star) * (q_P - q_M)
        )
    else:
        raise ValueError("flux_type must be 'central' or 'upwind'.")

    # Conservative interior physical line flux E(c q).
    cq_r_b = E @ (c_r * q)
    cq_s_b = E @ (c_s * q)
    F_cons_b = nr_f * cq_r_b + ns_f * cq_s_b
    F_cons_M = F_cons_b.T.flatten()[vmapM]

    # 4. H2: volume-compatible vs naive interior boundary flux
    if h2_mode == "A":
        if formulation == "divergence":
            F_minus = F_cons_M
        else:
            # Local speed a_M belongs to the split interior flux.
            # Common speed a_star belongs only to F_star.
            F_minus = 0.5 * F_cons_M + 0.5 * a_M * q_M
    else:
        # Deliberately incompatible product-trace ablation.
        F_minus = a_M * q_M

    penalty_b = F_minus - F_star

    # 5. H4: two algebraically equivalent projected-lift paths
    if h4_mode == "A":
        # L_f already contains W_f.
        lifted = L_f @ penalty_b
    else:
        # Equivalent implicit path:
        # V M^{-1} V^T E^T W_f penalty.
        scaled_penalty = penalty_b * face_w[:, jnp.newaxis]
        lifted = projected_lift_core @ (E.T @ scaled_penalty)

    # Correct strong-form sign:
    # q_t = -J^{-1} D F + J^{-1} L(F^- - F^*)
    surface_term = (1.0 / J) * lifted
    return volume_term + surface_term


In [5]:
# STREAMING_CHUNK: Configuring time stepper and assembly functions...
def run_ablation_case(
    k_degree=4,
    n_div=4,
    t_final=120 * 86400.0,
    CFL=0.1,
    formulation="split3",
    flux_type="central",
    h1_mode="A",
    h2_mode="A",
    h3_mode="A",
    h4_mode="A",
):
    R_sphere = 6.37122e6
    REFERENCE_AREA = 2.0

    nodes, EToV = generate_spherical_octahedron_mesh(
        n_div,
        R_sphere=1.0,
    )
    K = EToV.shape[0]
    EToE, EToF = build_connectivity(EToV)

    ref_data = get_reference_data("table2", k_degree)
    xi_ref = ref_data["xi"]
    eta_ref = ref_data["eta"]
    weights_ref = ref_data["weights"]

    points_1d, weights_1d, xi_face, eta_face = get_face_quadrature_1d(
        k_degree
    )
    Np = len(xi_ref)
    nfp = len(weights_1d)

    V_nodal = vandermonde_2d_dubiner(
        xi_ref,
        eta_ref,
        k_degree,
    )
    Vr, Vs = grad_vandermonde_2d_dubiner(
        xi_ref,
        eta_ref,
        k_degree,
    )
    V_face = vandermonde_2d_dubiner(
        xi_face,
        eta_face,
        k_degree,
    )

    # Reference norm and projected operators
    W = np.diag(weights_ref)
    H = REFERENCE_AREA * W

    ortho_mat_W = V_nodal.T @ W @ V_nodal
    ortho_err = np.linalg.norm(
        ortho_mat_W - 0.5 * np.eye(V_nodal.shape[1])
    )
    assert ortho_err < 1e-14, (
        f"Table 2 orthogonality check failed: {ortho_err}"
    )
    print(
        "  [Sanity Check Passed] "
        f"Table 2 Modal Orthogonality Error: {ortho_err:.2e}"
    )

    M_modal = V_nodal.T @ H @ V_nodal
    projection = np.linalg.solve(
        M_modal,
        V_nodal.T @ H,
    )

    E = V_face @ projection
    D_r_ref = Vr @ projection
    D_s_ref = Vs @ projection

    # Correct projected lift:
    # L_f = V M^{-1} V_f^T W_f, M = V^T H V, H = 2W.
    W_f = np.diag(np.tile(weights_1d, 3))
    L_f = V_nodal @ np.linalg.solve(
        M_modal,
        V_face.T @ W_f,
    )

    # Equivalent implicit lift core used only by H4-B:
    # (V M^{-1} V^T) E^T W_f = V M^{-1} V_f^T W_f.
    projected_lift_core = V_nodal @ np.linalg.solve(
        M_modal,
        V_nodal.T,
    )

    vmapM, vmapP, face_is_boundary = build_face_global_index_maps(
        EToV,
        EToE,
        EToF,
        K,
        nfp,
    )

    X, Y, Z = compute_global_coordinates(
        nodes,
        EToV,
        xi_ref,
        eta_ref,
        R_sphere,
    )
    (
        J,
        a1x,
        a1y,
        a1z,
        a2x,
        a2y,
        a2z,
        exact_c_r_b,
        exact_c_s_b,
    ) = compute_exact_metrics_3d(
        nodes,
        EToV,
        xi_ref,
        eta_ref,
        xi_face,
        eta_face,
    )

    if np.any(J <= 0.0):
        raise ValueError(
            "Non-positive radial-projection Jacobian detected. "
            "Check element orientation."
        )

    u0 = (2 * np.pi * 1.0) / (12 * 86400)
    alpha0 = np.pi / 4.0
    U_3d, V_3d, W_3d = velocity_solid_body_3d(
        X / R_sphere,
        Y / R_sphere,
        Z / R_sphere,
        u0,
        alpha0,
    )
    c_r_np = J * (
        a1x * U_3d + a1y * V_3d + a1z * W_3d
    )
    c_s_np = J * (
        a2x * U_3d + a2y * V_3d + a2z * W_3d
    )
    face_w = np.tile(weights_1d, 3)

    # JAX constants are created once and reused by every RHS evaluation.
    rhs_args = (
        jnp.array(D_r_ref),
        jnp.array(D_s_ref),
        jnp.array(E),
        jnp.array(L_f),
        jnp.array(J),
        jnp.array(vmapM),
        jnp.array(vmapP),
        jnp.array(face_is_boundary),
        jnp.array(face_w),
        jnp.array(projected_lift_core),
        jnp.array(c_r_np),
        jnp.array(c_s_np),
        jnp.array(exact_c_r_b),
        jnp.array(exact_c_s_b),
    )

    def eval_rhs(q_value):
        return compute_rhs_ablation(
            q_value,
            *rhs_args,
            flux_type=flux_type,
            formulation=formulation,
            h1_mode=h1_mode,
            h2_mode=h2_mode,
            h3_mode=h3_mode,
            h4_mode=h4_mode,
        )

    # 1. Initial state and mass diagnostics
    Q_init = exact_gaussian_bell_3d_jax(
        X / R_sphere,
        Y / R_sphere,
        Z / R_sphere,
        0.0,
        u0,
        1.0,
        alpha0,
    )

    # Physical quadrature includes reference area |T| = 2.
    M_diag = (
        REFERENCE_AREA
        * weights_ref[:, np.newaxis]
        * J
    )
    mass_0 = float(np.sum(np.array(Q_init) * M_diag))

    rhs_0 = eval_rhs(jnp.array(Q_init))
    dM_dt_0 = float(np.sum(np.array(rhs_0) * M_diag))

    # 2. Semi-discrete matrix and operator-level diagnostics
    total_dof = Np * K
    A_mat = np.zeros((total_dof, total_dof))

    for i in range(total_dof):
        q_e = np.zeros(total_dof)
        q_e[i] = 1.0
        q_in = jnp.array(
            q_e.reshape((Np, K), order="F")
        )
        rhs_e = eval_rhs(q_in)
        A_mat[:, i] = np.array(rhs_e).flatten(order="F")

    mass_row = M_diag.flatten(order="F")
    mass_operator_residual = float(
        np.linalg.norm(mass_row @ A_mat, ord=np.inf)
    )

    eigvals = la.eigvals(A_mat)
    max_re_lambda = float(np.max(np.real(eigvals)))

    # 3. LSRK54 integration
    h_min = float(np.min(compute_h_min(nodes, EToV)))
    dt_global = (
        CFL
        * h_min
        / (u0 * (k_degree + 1) ** 2)
    )
    dt_sample = 86400.0
    total_sample_steps = int(np.ceil(t_final / dt_sample))
    sub_steps = int(
        np.ceil(
            (t_final / total_sample_steps)
            / dt_global
        )
    )
    dt_rk = (
        (t_final / total_sample_steps)
        / sub_steps
    )

    A_RK = jnp.array(
        [
            0.0,
            -567301805773.0 / 1357537059087.0,
            -2404267990393.0 / 2016746695238.0,
            -3550918686646.0 / 2091501179385.0,
            -1275806237668.0 / 842570457699.0,
        ]
    )
    B_RK = jnp.array(
        [
            1432997174477.0 / 9575080441755.0,
            5161836677717.0 / 13612068292357.0,
            1720146321549.0 / 2090206949498.0,
            3134564353537.0 / 4481467310338.0,
            2277821191437.0 / 14882151754819.0,
        ]
    )

    Q_curr = jnp.array(Q_init)
    t_curr = 0.0
    diverged = False
    completed_steps = 0

    for step in range(total_sample_steps):
        for sub in range(sub_steps):
            du = jnp.zeros_like(Q_curr)
            Q_stage = Q_curr

            for stage in range(5):
                R_Q = eval_rhs(Q_stage)
                du = A_RK[stage] * du + dt_rk * R_Q
                Q_stage = Q_stage + B_RK[stage] * du

            Q_curr = Q_stage
            t_curr += dt_rk
            completed_steps += 1

            q_abs_max = float(jnp.max(jnp.abs(Q_curr)))
            if (
                bool(jnp.isnan(Q_curr).any())
                or bool(jnp.isinf(Q_curr).any())
                or q_abs_max > 1e10
            ):
                diverged = True
                break

        if diverged:
            break

    if diverged:
        final_mass_err = np.nan
        final_l2_err = np.nan
        h4_lift_difference = np.nan
    else:
        Q_final = np.array(Q_curr)
        mass_T = float(np.sum(Q_final * M_diag))
        final_mass_err = abs(mass_T - mass_0) / max(
            abs(mass_0),
            np.finfo(float).eps,
        )

        Q_exact_T = np.array(
            exact_gaussian_bell_3d_jax(
                X / R_sphere,
                Y / R_sphere,
                Z / R_sphere,
                t_curr,
                u0,
                1.0,
                alpha0,
            )
        )
        err_field = Q_final - Q_exact_T
        final_l2_err = (
            np.sqrt(np.sum(err_field**2 * M_diag))
            / np.sqrt(np.sum(Q_exact_T**2 * M_diag))
        )

        # H4-A and H4-B should be equivalent after scaling correction.
        sample_penalty = np.sin(
            np.arange(3 * nfp * K, dtype=float)
        ).reshape((3 * nfp, K), order="F")
        lift_a = L_f @ sample_penalty
        lift_b = projected_lift_core @ (
            E.T @ (face_w[:, None] * sample_penalty)
        )
        h4_lift_difference = float(
            np.linalg.norm(lift_a - lift_b, ord=np.inf)
        )

    return {
        "dM_dt_0": dM_dt_0,
        "mass_operator_residual": mass_operator_residual,
        "max_re_lambda": max_re_lambda,
        "final_mass_err": final_mass_err,
        "final_l2_err": final_l2_err,
        "h4_lift_difference": h4_lift_difference,
        "diverged": diverged,
        "nsteps": completed_steps,
        "dt_rk": float(dt_rk),
        "actual_final_time": float(t_curr),
    }


## Corrected Ablation Study Matrix

Run the cell below after executing all setup and definition cells.

The baseline is `Group 0 (Full Alignment)`. Groups 1–4 change one modeling
choice at a time. Group 4 checks that the explicit and implicit projected-lift
paths are algebraically equivalent after the reference-area correction.


In [6]:
# STREAMING_CHUNK: Executing corrected ablation matrix runs...
results_table = []

exp_groups = [
    {
        "id": "Group 0 (Full Alignment)",
        "h1": "A",
        "h2": "A",
        "h3": "A",
        "h4": "A",
    },
    {
        "id": "Group 1 (Exact Face Metric)",
        "h1": "B",
        "h2": "A",
        "h3": "A",
        "h4": "A",
    },
    {
        "id": "Group 2 (Naive Penalty)",
        "h1": "A",
        "h2": "B",
        "h3": "A",
        "h4": "A",
    },
    {
        "id": "Group 3 (One-Sided Interface Speed)",
        "h1": "A",
        "h2": "A",
        "h3": "B",
        "h4": "A",
    },
    {
        "id": "Group 4 (Implicit Equivalent Lift)",
        "h1": "A",
        "h2": "A",
        "h3": "A",
        "h4": "B",
    },
]

formulations = [
    "split3",
    "split2",
    "divergence",
]

for grp in exp_groups:
    print(
        "\n================ "
        f"Executing {grp['id']} "
        "================"
    )

    for form in formulations:
        t0 = time.time()

        res = run_ablation_case(
            k_degree=4,
            n_div=4,
            t_final=12 * 86400.0,
            CFL=0.1,
            formulation=form,
            flux_type="central",
            h1_mode=grp["h1"],
            h2_mode=grp["h2"],
            h3_mode=grp["h3"],
            h4_mode=grp["h4"],
        )

        res_row = {
            "Group": grp["id"],
            "Formulation": form,
            "dM/dt (t=0)": f"{res['dM_dt_0']:.4e}",
            "||m^T A||_inf": f"{res['mass_operator_residual']:.4e}",
            "Max Re(lambda)": f"{res['max_re_lambda']:.6e}",
            "H4 A-B Diff": (
                f"{res['h4_lift_difference']:.4e}"
                if not np.isnan(res["h4_lift_difference"])
                else "DIVERGED"
            ),
            "Final Mass Err": (
                f"{res['final_mass_err']:.4e}"
                if not np.isnan(res["final_mass_err"])
                else "DIVERGED"
            ),
            "Final L2 Err": (
                f"{res['final_l2_err']:.4e}"
                if not np.isnan(res["final_l2_err"])
                else "DIVERGED"
            ),
            "Steps": res["nsteps"],
            "dt": f"{res['dt_rk']:.6e}",
            "Time (s)": f"{time.time() - t0:.2f}",
        }
        results_table.append(res_row)

        print(
            f"  [{form:10s}] "
            f"dM/dt={res_row['dM/dt (t=0)']} | "
            f"||m^T A||={res_row['||m^T A||_inf']} | "
            f"Max Re(lam)={res_row['Max Re(lambda)']} | "
            f"Mass Err={res_row['Final Mass Err']}"
        )

df_res = pd.DataFrame(results_table)

print(
    "\n================ "
    "CORRECTED ABLATION STUDY SUMMARY TABLE "
    "================"
)
print(df_res.to_string(index=False))

OUTPUT_CSV = START_DIR / "test3_corrected_ablation_study_results.csv"
df_res.to_csv(OUTPUT_CSV, index=False)
print("\nSaved:", OUTPUT_CSV)



================ Executing Group 0 (Full Alignment) ================
  [Sanity Check Passed] Table 2 Modal Orthogonality Error: 2.52e-15
  [split3    ] dM/dt=-6.6769e-24 | ||m^T A||=3.3881e-21 | Max Re(lam)=1.861516e-19 | Mass Err=8.9831e-16
  [Sanity Check Passed] Table 2 Modal Orthogonality Error: 2.52e-15
  [split2    ] dM/dt=-3.4135e-23 | ||m^T A||=1.8523e-11 | Max Re(lam)=2.027517e-19 | Mass Err=5.5815e-08
  [Sanity Check Passed] Table 2 Modal Orthogonality Error: 2.52e-15
  [divergence] dM/dt=-2.5281e-23 | ||m^T A||=3.1764e-21 | Max Re(lam)=3.851336e-08 | Mass Err=1.7966e-16

================ Executing Group 1 (Exact Face Metric) ================
  [Sanity Check Passed] Table 2 Modal Orthogonality Error: 2.52e-15
  [split3    ] dM/dt=2.3820e-23 | ||m^T A||=2.3197e-08 | Max Re(lam)=2.963578e-19 | Mass Err=2.7669e-03
  [Sanity Check Passed] Table 2 Modal Orthogonality Error: 2.52e-15
  [split2    ] dM/dt=2.7077e-23 | ||m^T A||=2.3195e-08 | Max Re(lam)=1.922584e-19 | Mass Err=2.767